In [1]:
import warnings; warnings.simplefilter("ignore", FutureWarning)  # Suppress deprecation warnings
from pyspark.sql.functions import col, count, isnan, regexp_extract, when # Data transformation functions
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler  # Feature engineering transformers
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, MultilayerPerceptronClassifier  # ML Classifiers
from pyspark.ml.evaluation import MulticlassClassificationEvaluator  # Model evaluation metrics
import pandas as pd, plotly.express as px

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LendingClub") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 00:02:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.setLogLevel("ERROR") # Disable WARN logs from Spark's underlying Java engine
spark.conf.set("spark.sql.ansi.enabled", "false") # Kaggle compatibility setting

In [4]:
cols = [
    "id", "purpose", "term", "verification_status", "acc_now_delinq",
    "addr_state", "annual_inc", "application_type", "dti", "grade",
    "home_ownership", "initial_list_status", "installment", "int_rate",
    "loan_amnt", "loan_status", "tax_liens", "delinq_amnt", "policy_code",
    "last_fico_range_high", "last_fico_range_low", "recoveries", "collection_recovery_fee"]

# Load dataset with schema inference & select columns natively
df = (
    spark.read.csv(
        "/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz",
        header=True,
        inferSchema=True,
    )
    .select(cols) # check null df.pandas_api().isnull().sum()
    .dropna()  # Clean nulls directly on read
)
# Preview using native Spark limit (avoids pandas conversion/warnings)
df.limit(5).toPandas()

,id,purpose,term,verification_status,acc_now_delinq,addr_state,annual_inc,application_type,dti,grade,...,int_rate,loan_amnt,loan_status,tax_liens,delinq_amnt,policy_code,last_fico_range_high,last_fico_range_low,recoveries,collection_recovery_fee
0,68407277,debt_consolidation,36 months,Not Verified,0.0,PA,55000.0,Individual,5.91,C,...,13.99,3600.0,Fully Paid,0.0,0.0,1.0,564.0,560.0,0.0,0.0
1,68355089,small_business,36 months,Not Verified,0.0,SD,65000.0,Individual,16.06,C,...,11.99,24700.0,Fully Paid,0.0,0.0,1.0,699.0,695.0,0.0,0.0
2,68341763,home_improvement,60 months,Not Verified,0.0,IL,63000.0,Joint App,10.78,B,...,10.78,20000.0,Fully Paid,0.0,0.0,1.0,704.0,700.0,0.0,0.0
3,66310712,debt_consolidation,60 months,Source Verified,0.0,NJ,110000.0,Individual,17.06,C,...,14.85,35000.0,Current,0.0,0.0,1.0,679.0,675.0,0.0,0.0
4,68476807,major_purchase,60 months,Source Verified,0.0,PA,104433.0,Individual,25.37,F,...,22.45,10400.0,Fully Paid,0.0,0.0,1.0,704.0,700.0,0.0,0.0


# EDA
## 'purpose'

In [5]:
df_with_count = df.groupBy("purpose").count()
df_with_count.toPandas()

,purpose,count
0,educational,404
1,home_improvement,150290
2,moving,15369
3,other,139270
4,house,14119
5,wedding,2351
6,small_business,24638
7,credit_card,516570
8,vacation,15518
9,car,23996


In [6]:
# Replacing values in the 'purpose' column based on the 'count' column condition
# If 'count' is less than 300, set 'purpose' to "other", else keep the original 'purpose'
df = df.join(df_with_count, 'purpose').withColumn("purpose", when(col("count") < 300, "other").otherwise(col("purpose"))).drop('count')

In [7]:
unique_purposes = df.select("purpose").distinct()
unique_purposes.toPandas()

,purpose
0,educational
1,home_improvement
2,moving
3,other
4,house
5,wedding
6,small_business
7,credit_card
8,vacation
9,car


## 'term'

In [8]:
df.groupby('term').count().toPandas()

,term,count
0,60 months,650191
1,36 months,1608405


In [9]:
# Extract the numeric digits and cast them to integers instantly
df = df.withColumn("term", regexp_extract(col("term"), r"\d+", 0).cast("int"))
df.groupby('term').count().toPandas()

,term,count
0,36,1608405
1,60,650191


## 'verification'

In [10]:
df.groupby('verification_status').count().toPandas()

,verification_status,count
0,Verified,629395
1,Not Verified,743060
2,Source Verified,886141


In [11]:
# If 'verification_status' is either "Verified" or "Source Verified", set 'verification_status_encoded' to 0# Otherwise, set it to 1
# Transform and drop column in a single line
# Create the binary flag and drop the old string column instantly
df = df.withColumn("verification_status_encoded", when(col("verification_status").isin(["Verified", "Source Verified"]), 0).otherwise(1)).drop("verification_status")

In [12]:
# Verify counts using the encoded column name in a single line
df.groupBy("verification_status_encoded").count().toPandas()

,verification_status_encoded,count
0,1,743060
1,0,1515536


## 'acc_now_delinq'

In [13]:
df.groupby('acc_now_delinq').count().toPandas()

,acc_now_delinq,count
0,7.0,1
1,1.0,8290
2,5.0,3
3,0.0,2249817
4,4.0,11
5,3.0,50
6,14.0,1
7,2.0,421
8,6.0,2


In [14]:
# 1. Update the column: Convert decimals to whole integers, and cap any extreme values at 4
df = df.withColumn('acc_now_delinq', col('acc_now_delinq').cast('int')) \
       .withColumn('acc_now_delinq', when(col('acc_now_delinq') >= 4, 4).otherwise(col('acc_now_delinq'))) \
       .filter(col('acc_now_delinq').between(0, 4)) # Keep only valid records between 0 and 4

In [15]:
# 2. Shortest standard check: Display the clean distribution without needing .show()
df.groupBy('acc_now_delinq').count().toPandas()

,acc_now_delinq,count
0,1,8290
1,3,50
2,2,421
3,4,18
4,0,2249817


## 'application_type'

In [16]:
df.groupby('application_type').count().toPandas()

,application_type,count
0,Individual,2139597
1,Joint App,118999


In [17]:
# 1. Map text categories to integers (unmapped rows automatically become NULL)
df = df.withColumn('application_type', 
                   when(col('application_type') == 'Joint App', 0)
                   .when(col('application_type') == 'Individual', 1)) \
       .filter(col('application_type').isNotNull()) # Drop rows that didn't match our valid values

In [18]:
# 2. Shortest standard check: Verify the clean 0 and 1 distribution instantly
df.groupBy('application_type').count().toPandas()

,application_type,count
0,1,2139597
1,0,118999


## 'grade'

In [19]:
df.groupby('grade').count().toPandas()

,grade,count
0,G,12144
1,C,649471
2,B,663013
3,A,432662
4,E,135506
5,F,41758
6,D,324042


In [20]:
# Create a StringIndexer to convert 'grade' column into numerical indices
df = StringIndexer(inputCol="grade", outputCol="grade_index", stringOrderType="alphabetAsc").fit(df).transform(df).drop("grade")

In [21]:
df.groupby('grade_index').count().toPandas()

,grade_index,count
0,6.0,12144
1,1.0,663013
2,4.0,135506
3,0.0,432662
4,3.0,324042
5,2.0,649471
6,5.0,41758


## 'Feature Encoding'

In [23]:
# 1. Define column names
raw_string_cols = ["purpose", "addr_state", "home_ownership", "initial_list_status"]
index_cols = ["purpose_idx", "addr_state_idx", "home_ownership_idx", "initial_list_status_idx"]
vec_cols = ["purpose_vec", "addr_state_vec", "home_ownership_vec", "initial_list_status_vec"]

# 2. StringIndexer: Raw Strings -> _idx
indexer = StringIndexer(inputCols=raw_string_cols, outputCols=index_cols)
df = indexer.fit(df).transform(df)

# 3. OneHotEncoder: _idx -> _vec
encoder = OneHotEncoder(inputCols=index_cols, outputCols=vec_cols)
df = encoder.fit(df).transform(df)

# 4. Clean up: Drop raw strings AND intermediate _idx columns
df = df.drop(*raw_string_cols, *index_cols)

# 5. Inspect your updated schema
df.printSchema()  # Print clean schema

root
 |-- id: string (nullable = true)
 |-- term: integer (nullable = true)
 |-- acc_now_delinq: integer (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- application_type: integer (nullable = true)
 |-- dti: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- tax_liens: double (nullable = true)
 |-- delinq_amnt: double (nullable = true)
 |-- policy_code: string (nullable = true)
 |-- last_fico_range_high: string (nullable = true)
 |-- last_fico_range_low: string (nullable = true)
 |-- recoveries: string (nullable = true)
 |-- collection_recovery_fee: string (nullable = true)
 |-- verification_status_encoded: integer (nullable = false)
 |-- grade_index: double (nullable = false)
 |-- purpose_vec: vector (nullable = true)
 |-- addr_state_vec: vector (nullable = true)
 |-- home_ownership_vec: vector (nullable = true)
 |-- initial_

## 'loan_status' -> Target 

In [24]:
df.groupby('loan_status').count().toPandas()

,loan_status,count
0,Fully Paid,1076218
1,Late (31-120 days),21443
2,In Grace Period,8427
3,Late (16-30 days),4344
4,Default,40
5,Does not meet the credit policy. Status:Fully ...,1913
6,Charged Off,268452
7,Does not meet the credit policy. Status:Charge...,741
8,Current,877018


## Target Encoding & Downsampling

In [25]:
# 1. Encode target: 0 = Fully Paid, 1 = Late / Grace Period, 2 = Charged Off / Default
df_target = df.withColumn(
    "target",
    when(col("loan_status") == "Fully Paid", 0)
    .when(col("loan_status").isin("Late (16-30 days)", "Late (31-120 days)", "In Grace Period"), 1)
    .when(col("loan_status").isin("Charged Off", "Default"), 2)
).filter(col("target").isNotNull()).drop("loan_status")

# 2. Downsample Class 0 to 30% and save to df_downsampled
df_downsampled = df_target.sampleBy("target", fractions={0: 0.3, 1: 1.0, 2: 1.0}, seed=42)

# 3. Check downsampled counts
df_downsampled.groupBy("target").count().toPandas()

,target,count
0,1,34214
1,2,268492
2,0,324114


In [26]:
# 1. Combine counts into one dataframe
df_all = pd.concat([
    df_target.groupBy("target").count().toPandas().assign(Stage="Before"),
    df_downsampled.groupBy("target").count().toPandas().assign(Stage="After")
])

# 2. Plot side-by-side charts using facet_col
px.bar(df_all, x="target", y="count", facet_col="Stage", width=700, height=350)

## Feature Vector Assembly

In [28]:
# 1. Identify feature columns
feature_cols = [c for c in df_downsampled.columns if c not in ["id", "target"]]

# 2. Cast non-vector feature columns to double to prevent type errors
df_downsampled = df_downsampled.select([
    col(c).cast("double") if not c.endswith("_vec") else col(c) 
    for c in df_downsampled.columns
])

# 3. Assemble features (86 features guaranteed)
df_downsampled = VectorAssembler(inputCols=feature_cols, outputCol="features_to_scale").transform(df_downsampled)
df_downsampled.select("features_to_scale", "target").show(5)

+--------------------+------+
|   features_to_scale|target|
+--------------------+------+
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   2.0|
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   0.0|
+--------------------+------+
only showing top 5 rows


In [29]:
print(f"Total columns: {len(feature_cols)}\nColumns: {feature_cols}")

Total columns: 21
Columns: ['term', 'acc_now_delinq', 'annual_inc', 'application_type', 'dti', 'installment', 'int_rate', 'loan_amnt', 'tax_liens', 'delinq_amnt', 'policy_code', 'last_fico_range_high', 'last_fico_range_low', 'recoveries', 'collection_recovery_fee', 'verification_status_encoded', 'grade_index', 'purpose_vec', 'addr_state_vec', 'home_ownership_vec', 'initial_list_status_vec']


## Data Splitting & Scaling

In [30]:
# 1. Split data (80% Train, 10% Test, 10% Validation)
train_data, temp_data = df_downsampled.randomSplit([0.8, 0.2], seed=42)
test_data, val_data = temp_data.randomSplit([0.5, 0.5], seed=42)

# 2. Fit MinMaxScaler on train set and transform all splits directly
scaler = MinMaxScaler(inputCol="features_to_scale", outputCol="features").fit(train_data)
train_data, val_data, test_data = scaler.transform(train_data), scaler.transform(val_data), scaler.transform(test_data)

## Model Training & Evaluation

In [32]:
# 1. Pre-load data splits directly into RAM
for ds in [train_data, val_data, test_data]: ds.cache().count()

# 2. Shared Evaluator instance
evaluator = MulticlassClassificationEvaluator(labelCol="target")

# 3. Model training & evaluation loop
def evaluate(model, name):
    fit_m = model.fit(train_data)
    res = {"Model": name}
    for label, ds in [("Train", train_data), ("Validation", val_data), ("Test", test_data)]:
        preds = fit_m.transform(ds)  # Transform once per split
        res[f"Accuracy ({label})"] = round(evaluator.setMetricName("accuracy").evaluate(preds), 3)
        res[f"F1 Score ({label})"] = round(evaluator.setMetricName("f1").evaluate(preds), 3)
    return res

models = [
    (LogisticRegression(featuresCol="features", labelCol="target"), "LogisticRegression"),
    (RandomForestClassifier(featuresCol="features", labelCol="target"), "Random Forest"),
    (MultilayerPerceptronClassifier(layers=[86, 64, 32, 3], featuresCol="features", labelCol="target", maxIter=20, seed=123), "Neural Network")
]

pd.DataFrame([evaluate(m, name) for m, name in models])

,Model,Accuracy (Train),F1 Score (Train),Accuracy (Validation),F1 Score (Validation),Accuracy (Test),F1 Score (Test)
0,LogisticRegression,0.879,0.861,0.881,0.864,0.877,0.859
1,Random Forest,0.866,0.842,0.869,0.845,0.865,0.840
2,Neural Network,0.752,0.730,0.752,0.731,0.750,0.728
